# CRA-GQA — Kaggle Inference Smoke Test

Fully self-contained on Kaggle. Clones repo, builds synthetic data, trains 1 epoch if needed, then runs `pipeline.infer()`.

**Settings:** GPU + Internet ON | **Run cells top-to-bottom**

**GPU note:** T4 is fastest to set up. **P100 is supported** (cell 1 auto-installs PyTorch 2.4+cu118). In Kaggle: Settings → Accelerator → GPU.

**Note:** Metrics on synthetic data are not meaningful. This only verifies the inference path runs end-to-end.

In [ ]:
# CELL 1 — numpy + GPU-matched PyTorch (do NOT import torch before this)
import json
import subprocess
import sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.0.1"], check=True)

def _gpu_cc_major():
    try:
        cc = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            text=True,
        ).strip().split(".")[0]
        return int(cc)
    except Exception:
        return 7

cc_major = _gpu_cc_major()
print("GPU compute capability major:", cc_major)

if cc_major < 7:
    # Pascal P100 (sm_60) — modern cu124 wheels drop sm_60 support
    cfg = {
        "index": "https://download.pytorch.org/whl/cu118",
        "torch": "2.4.1", "vision": "0.19.1", "audio": "2.4.1",
        "pyg_tag": "cu118",
    }
    print("Legacy GPU (e.g. P100) -> PyTorch 2.4.1+cu118")
else:
    cfg = {
        "index": "https://download.pytorch.org/whl/cu124",
        "torch": "2.5.1", "vision": "0.20.1", "audio": "2.5.1",
        "pyg_tag": "cu124",
    }
    print("Modern GPU (e.g. T4) -> PyTorch 2.5.1+cu124")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    f"torch=={cfg['torch']}", f"torchvision=={cfg['vision']}", f"torchaudio=={cfg['audio']}",
    "--index-url", cfg["index"],
], check=True)

Path("/kaggle/working/_torch_smoke_cfg.json").write_text(json.dumps(cfg))

import numpy as np
print("numpy:", np.__version__)
assert hasattr(np, "_no_nep50_warning"), "Restart session and rerun from cell 1"
print("PyTorch installed. Run cell 2 next.")

In [ ]:
# CELL 2 — clone repo, install deps, patch upstream issues
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
import torch

os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle notebook settings (not TPU).")
_probe = torch.randn(32, 32, device="cuda")
(_probe @ _probe).sum().item()
torch.cuda.synchronize()
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "| cap:", torch.cuda.get_device_capability(0))

REPO_ROOT = Path("/kaggle/working/CRA-GQA")
if not (REPO_ROOT / "main.py").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/WissingChen/CRA-GQA.git", str(REPO_ROOT)],
        check=True,
    )
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.47.1", "h5py==3.12.1", "pyyaml==6.0.2", "tabulate==0.9.0",
    "dominate==2.9.1",
], check=True)
cfg = json.loads(Path("/kaggle/working/_torch_smoke_cfg.json").read_text())
torch_v = torch.__version__.split("+")[0]
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyg_lib", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv",
    "-f", f"https://data.pyg.org/whl/torch-{torch_v}+{cfg['pyg_tag']}.html",
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "torch-geometric==2.6.1", "--no-deps",
], check=True)

import modules.m_tempclip.language_model as language_model
from transformers import RobertaConfig, RobertaModel
import torch.nn as nn

def _roberta_bert_init(self, tokenizer, lan):
    nn.Module.__init__(self)
    self.tokenizer = tokenizer
    if lan != "RoBERTa":
        raise ValueError(lan)
    config = RobertaConfig.from_pretrained("roberta-base", output_hidden_states=True)
    self.bert = RobertaModel.from_pretrained("roberta-base", config=config)

language_model.Bert.__init__ = _roberta_bert_init

# PyTorch 2.6+ defaults weights_only=True; causal pickles need False
_torch_load = torch.load
def _load_trusted(path, *args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _torch_load(path, *args, **kwargs)
torch.load = _load_trusted

import pipeline.base as base_pipeline

def _print_best_to_file(self):
    crt_time = time.asctime(time.localtime(time.time()))
    self.best_recorder["val"]["time"] = crt_time
    self.best_recorder["test"]["time"] = crt_time
    self.best_recorder["val"]["seed"] = self.cfgs["misc"]["seed"]
    self.best_recorder["test"]["seed"] = self.cfgs["misc"]["seed"]
    record_dir = self.cfgs["stat"]["record_dir"]
    os.makedirs(record_dir, exist_ok=True)
    record_path = os.path.join(record_dir, self.cfgs["dataset"]["name"] + ".csv")
    record_table = pd.read_csv(record_path) if os.path.exists(record_path) else pd.DataFrame()
    record_table = pd.concat(
        [record_table, pd.DataFrame([self.best_recorder["val"], self.best_recorder["test"]])],
        ignore_index=True,
    )
    record_table.to_csv(record_path, index=False)

def _print_best(self):
    metric = self.cfgs["stat"]["monitor"]["metric"]
    for split in ["val", "test"]:
        print(f"Best {split}:", self.best_recorder[split])

base_pipeline.BasePipeline._print_best_to_file = _print_best_to_file
base_pipeline.BasePipeline._print_best = _print_best
print("Repo:", REPO_ROOT)

In [ ]:
# CELL 3 — synthetic data + config
import json
import h5py
import numpy as np
import pandas as pd
import torch
import yaml
from torch_geometric.data import Data

from pipeline import pipeline_fns
from utils.misc import setup_seed

SMOKE_SEED = 42

def _sample_rows(n_videos=2, n_q_per_video=4):
    choices = [
        "A person walks", "A car drives", "A dog runs",
        "Nothing happens", "A bird flies",
    ]
    rows = []
    for vi in range(n_videos):
        vid = f"vid{vi:03d}"
        for qi in range(n_q_per_video):
            qid = f"q{qi:03d}"
            ans_idx = (vi * n_q_per_video + qi) % len(choices)
            answer = choices[ans_idx]
            rows.append({
                "video_id": vid,
                "question": f"What happens in video {vi} at step {qi}?",
                "answer": answer,
                "qid": qid,
                "type": "Tem",
                "a0": choices[0], "a1": choices[1], "a2": choices[2],
                "a3": choices[3], "a4": choices[4],
            })
    return pd.DataFrame(rows)

def setup_smoke_dataset(root, max_feats=8, seed=SMOKE_SEED):
    rng = np.random.default_rng(seed)
    root = Path(root)
    csv_dir = root / "nextgqa"
    feat_dir = root / "nextqa" / "video_feature" / "CLIP_L"
    causal_dir = csv_dir / "causal_feature"
    feat_dir.mkdir(parents=True, exist_ok=True)
    train_df, val_df, test_df = _sample_rows(3, 4), _sample_rows(1, 4), _sample_rows(1, 4)
    csv_dir.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(csv_dir / "train.csv", index=False)
    val_df.to_csv(csv_dir / "val.csv", index=False)
    test_df.to_csv(csv_dir / "test.csv", index=False)
    n_frames = max(max_feats, 16)
    for df, name in [(train_df, "train"), (val_df, "val"), (test_df, "test")]:
        vids = sorted(df["video_id"].astype(str).unique())
        with h5py.File(feat_dir / f"{name}.h5", "w") as fp:
            fp.create_dataset("vid", data=np.array(vids, dtype="S"))
            fp.create_dataset("CLIPL_I", data=rng.random((len(vids), n_frames, 768), dtype=np.float32))
    for split, df in [("val", val_df), ("test", test_df)]:
        gsub = {}
        for _, row in df.iterrows():
            vid, qid = str(row["video_id"]), str(row["qid"])
            gsub.setdefault(vid, {"duration": 30.0, "fps": 30.0, "location": {}})
            gsub[vid]["location"][qid] = [[5.0, 15.0]]
        times = {v: np.linspace(0, 30, n_frames).tolist() for v in sorted(df["video_id"].astype(str).unique())}
        json.dump(gsub, open(csv_dir / f"gsub_{split}.json", "w"))
        json.dump(times, open(csv_dir / f"frame2time_{split}.json", "w"))
    causal_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    x = torch.randn(32, 768)
    edge = torch.tensor([[i, (i + 1) % 32] for i in range(32)], dtype=torch.long).t().contiguous()
    torch.save({"k_center": Data(x=x, edge_index=edge)}, causal_dir / "qa_graphs.npy")
    torch.save({"k_center": torch.randn(16, 768)}, causal_dir / "visual_clusters.npy")

def load_smoke_config(cfg_path, running_name, record_dir="/kaggle/working/output", resume=None):
    with open(cfg_path) as f:
        cfgs = yaml.safe_load(f)
    cfgs["dataset"].update(batch_size=2, num_thread_reader=0, max_feats=8)
    cfgs["optim"].update(epochs=1, save_period=1, step_size=1)
    cfgs["stat"]["record_dir"] = record_dir
    cfgs["stat"]["resume"] = resume
    cfgs["stat"]["monitor"]["early_stop"] = 1
    cfgs["misc"].update(cuda="0", running_name=running_name, seed=SMOKE_SEED)
    cfgs["model"]["lan_weight_path"] = "roberta-base"
    cfgs["model"]["vg_loss"] = 0
    return cfgs

setup_seed(SMOKE_SEED)
setup_smoke_dataset(REPO_ROOT / "data", max_feats=8, seed=SMOKE_SEED)

RUN_NAME = "kaggle/infer_smoke"
OUTPUT_DIR = "/kaggle/working/output"
CKPT_DIR = Path(OUTPUT_DIR) / RUN_NAME / "checkpoint"
RESUME_CKPT = CKPT_DIR / "model_best.pth"

cfgs = load_smoke_config(REPO_ROOT / "config/CRA/CRA_NextGQA.yml", RUN_NAME, OUTPUT_DIR)
print("Data + config ready")

In [ ]:
# CELL 4 — mock train if no checkpoint (2 batches, NOT full epoch)
import time
import torch.nn as nn

MOCK_BATCHES = 2

def mock_train(pipeline, n_batches=MOCK_BATCHES):
    pipeline.model.train()
    print(f"Mock train: {n_batches} batch(es) only", flush=True)
    for i, batch in enumerate(pipeline.train_dataloader):
        if i >= n_batches:
            break
        t0 = time.time()
        print(f"  batch {i+1}/{n_batches} — forward+backward...", flush=True)
        loss, acc = pipeline.get_loss(batch)
        pipeline.optimizer.zero_grad()
        loss["total_loss"].backward()
        if pipeline.cfgs["optim"]["clip"]:
            nn.utils.clip_grad_norm_(pipeline.model.parameters(), pipeline.cfgs["optim"]["clip"])
        pipeline.optimizer.step()
        pipeline.lr_scheduler.step()
        print(f"  batch {i+1} OK | loss={float(loss['total_loss']):.4f} | {time.time()-t0:.1f}s", flush=True)
    ckpt_dir = Path(pipeline.cfgs["stat"]["record_dir"]) / pipeline.cfgs["misc"]["running_name"] / "checkpoint"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    state = {"epoch": 1, "state_dict": pipeline.model.state_dict(), "optimizer": pipeline.optimizer.state_dict(), "monitor_best": 0.0}
    torch.save(state, ckpt_dir / "current_checkpoint.pth")
    torch.save(state, ckpt_dir / "model_best.pth")
    return ckpt_dir / "model_best.pth"

TRAIN_SMOKE_DIR = Path(OUTPUT_DIR) / "kaggle/train_smoke/checkpoint"
candidates = [
    RESUME_CKPT,
    TRAIN_SMOKE_DIR / "model_best.pth",
    CKPT_DIR / "current_checkpoint.pth",
    TRAIN_SMOKE_DIR / "current_checkpoint.pth",
]
RESUME_CKPT = next((p for p in candidates if p.exists()), RESUME_CKPT)

if not RESUME_CKPT.exists():
    print("No checkpoint — mock train 2 batches (batch 1 có thể ~2-5 phút trên P100)...", flush=True)
    setup_seed(cfgs["misc"]["seed"])
    train_pipeline = pipeline_fns[cfgs["optim"]["pipeline"]](cfgs)
    RESUME_CKPT = mock_train(train_pipeline)
else:
    print("Using checkpoint:", RESUME_CKPT, flush=True)

assert RESUME_CKPT.exists(), f"Missing checkpoint. Tried: {candidates}"

In [ ]:
# CELL 5 — inference smoke test
cfgs_infer = load_smoke_config(
    REPO_ROOT / "config/CRA/CRA_NextGQA.yml",
    RUN_NAME + "_eval",
    OUTPUT_DIR,
    resume=str(RESUME_CKPT),
)
setup_seed(cfgs_infer["misc"]["seed"])
infer_pipeline = pipeline_fns[cfgs_infer["optim"]["pipeline"]](cfgs_infer)

scores = infer_pipeline.infer()
for key in ["val_acc", "test_acc"]:
    assert key in scores, f"Missing {key}"
    print(f"{key}: {scores[key]:.4f}")
print("Inference smoke test passed.")